In [1]:
%load_ext watermark


In [2]:
from IPython.display import Math, display
import numpy as np
import sympy as sp

from pylib.theory._calc_catchup_prob_closed import calc_catchup_prob_closed
from pylib.theory._calc_denovo_critical_potential_closed import (
    calc_denovo_critical_potential_closed,
)
from pylib.theory._calc_denovo_mutator_frequency import (
    calc_denovo_mutator_frequency,
)
from pylib.theory._calc_denovo_peak_prevalence_closed import (
    calc_denovo_peak_prevalence_closed,
)
from pylib.theory._calc_denovo_step_prob import calc_denovo_step_prob
from pylib.theory._calc_drift_fixprob import calc_drift_fixprob
from pylib.theory._calc_escape_prob_closed import calc_escape_prob_closed
from pylib.theory._calc_fill_time import calc_fill_time
from pylib.theory._calc_hitchhike_prob_closed import calc_hitchhike_prob_closed
from pylib.theory._calc_load_params import calc_load_params
from pylib.theory._calc_mutator_fixprob_closed import (
    calc_mutator_fixprob_closed,
)
from pylib.theory._calc_ncrit_closed import calc_ncrit_closed
from pylib.theory._calc_sweep_time import calc_sweep_time
from pylib.theory._calc_unswept_prob import calc_unswept_prob


In [3]:
%watermark -diwmuv -iv


Last updated: 2026-11-26T12:07:32.941913+00:00

Python implementation: CPython
Python version       : 3.10.22
IPython version      : 7.31.1

Compiler    : GCC 13.3.0
OS          : Linux
Release     : 7.0.0-1017-azure
Machine     : x86_64
Processor   : x86_64
CPU cores   : 4
Architecture: 64bit

sympy: 1.13.3
numpy: 2.1.2

Watermark: 2.4.3



# Symbolic derivations for the analytical model

This notebook works through, with [SymPy](https://www.sympy.org), each step of calculus or algebra behind the closed-form model of mutator fixation under limited adaptive potential: the differential equations solved, the integrals evaluated, and the equations inverted for critical population sizes.
Each section ends with assertions that the symbolic result matches the expression used in the manuscript, and the final section checks the expressions numerically against `pylib.theory` under simulation parameters.

Modeling assumptions (what is being integrated, and why) are stated in words; SymPy only certifies the mathematics that follows from them.


In [4]:
def show(lhs: str, expr, approx: bool = False) -> None:
    """Display `lhs = expr` as typeset math, or `lhs ≈ expr` for an
    approximation."""
    relation = r"\approx" if approx else "="
    display(Math(f"{lhs} {relation} {sp.latex(expr)}"))


def is_zero(expr) -> bool:
    """Whether an expression simplifies to zero, symbolically."""
    forms = (
        expr,
        expr.rewrite(sp.exp),
        sp.expand_log(expr, force=True),
        sp.powsimp(sp.powdenest(expr, force=True), force=True),
    )
    return any(sp.simplify(form) == 0 for form in forms)


In [5]:
t, tau, x, y = sp.symbols("t tau x y", positive=True)
N, K, M = sp.symbols("N K M", positive=True)
s, delta, r, p, q = sp.symbols("s delta r p q", positive=True)
lam_b, lam_d, mu = sp.symbols("lambda_b lambda_d mu", positive=True)
u, u_n, u_m = sp.symbols("u u_n u_m", positive=True)
x_0, xbar = sp.symbols(r"x_0 \bar{x}", positive=True)
n, t_f, T, T_n = sp.symbols("n t_f T T_n", positive=True)
rho, v, tau_0 = sp.symbols("rho v tau_0", positive=True)


## 1. Ingredients

### Indirect selection against mutators, $\delta$

An individual with deleterious mutation rate $\lambda$ produces unloaded offspring with probability $e^{-\lambda}$ (Poisson zero term).
$\delta$ is mutators' excess rate of producing loaded offspring.


In [6]:
delta_expr = sp.exp(-lam_d) - sp.exp(-M * lam_d)
show(r"\delta", delta_expr)
delta_series = sp.series(delta_expr, lam_d, 0, 2).removeO()
show(r"\delta", sp.factor(delta_series), approx=True)
assert is_zero(delta_series - (M - 1) * lam_d)


<IPython.core.display.Math object>

<IPython.core.display.Math object>

### Load-free fraction of mutators, $q_0$

Under rank-based selection an unloaded mutator beats a loaded one with advantage $s$, so the unloaded fraction $q$ gains $s\,q\,(1 - q)$ per generation by selection and loses $(1 - e^{-U})\,q$ by mutation, with $U = M \lambda_d$.
$q_0$ is the nonzero balance point.


In [7]:
U = sp.symbols("U", positive=True)
balance = s * q * (1 - q) - (1 - sp.exp(-U)) * q
(q0_expr,) = [sol for sol in sp.solve(balance, q) if sol != 0]
show("q_0", q0_expr)
assert is_zero(q0_expr - (1 - (1 - sp.exp(-U)) / s))

# agrees with the multiplicative-fitness form exp(-U / s) to first order
q0_series = sp.series(q0_expr, U, 0, 2).removeO()
haigh_series = sp.series(sp.exp(-U / s), U, 0, 2).removeO()
show(r"q_0", q0_series, approx=True)
assert is_zero(q0_series - haigh_series)


<IPython.core.display.Math object>

<IPython.core.display.Math object>

### Fixation probability under the diffusion approximation (Kimura 1962)

Both the establishment probability $p$ and the drift term $P_\mathrm{drift}$ come from one equation.
For a haploid allele at frequency $x$ with selection coefficient $\sigma$, the fixation probability $U(x)$ satisfies the backward equation

$$\frac{x (1 - x)}{2N}\, U''(x) + \sigma\, x (1 - x)\, U'(x) = 0, \qquad U(0) = 0, \quad U(1) = 1.$$


In [8]:
sigma = sp.symbols("sigma", real=True, nonzero=True)
Ufix = sp.Function("U")
backward = sp.Eq(Ufix(x).diff(x, 2) / (2 * N) + sigma * Ufix(x).diff(x), 0)
kimura = sp.dsolve(backward, Ufix(x), ics={Ufix(0): 0, Ufix(1): 1}).rhs
kimura_std = (1 - sp.exp(-2 * N * sigma * x)) / (1 - sp.exp(-2 * N * sigma))
assert is_zero(kimura - kimura_std)
show("U(x)", kimura_std)


<IPython.core.display.Math object>

**Establishment probability $p$**: a single copy ($x = 1/N$) with advantage $\sigma = s$, in the limit of large $N$.


In [9]:
p_expr = sp.limit(kimura_std.subs({sigma: s, x: 1 / N}), N, sp.oo)
show("p", sp.simplify(p_expr))
assert is_zero(p_expr - (1 - sp.exp(-2 * s)))
show(r"p", sp.series(p_expr, s, 0, 2).removeO(), approx=True)


<IPython.core.display.Math object>

<IPython.core.display.Math object>

## 2. Escape

### Drift

Loaded mutators at start frequency $x_0$ have disadvantage $\sigma = -\delta$.


In [10]:
P_drift = (sp.exp(2 * N * delta * x_0) - 1) / (sp.exp(2 * N * delta) - 1)
assert is_zero(kimura_std.subs({sigma: -delta, x: x_0}) - P_drift)
show(r"P_\mathrm{drift}", P_drift)


<IPython.core.display.Math object>

### Hitchhiking

Absent beneficial mutations, mutator frequency declines logistically, $\dot{x} = -\delta\, x (1 - x)$.
While it does, the $N x(t)$ mutators found established beneficial lineages at rate $u_m$ each.
$P_\mathrm{hitch}$ is the probability of at least one such founding: one minus the Poisson zero term with mean $N u_m \int_0^\infty x(t)\, dt$.


In [11]:
X = sp.Function("x")
decline = sp.dsolve(
    sp.Eq(X(t).diff(t), -delta * X(t) * (1 - X(t))), X(t), ics={X(0): x_0}
).rhs
show("x(t)", decline)

exposure = sp.simplify(sp.integrate(decline, (t, 0, sp.oo), conds="none"))
show(r"\int_0^\infty x(t)\, dt", exposure)
assert is_zero(exposure - sp.log(1 / (1 - x_0)) / delta)

N_lower = delta / u_m
P_hitch = 1 - (1 - x_0) ** (N / N_lower)
assert is_zero(sp.exp(-N * u_m * exposure) - (1 - P_hitch))
show(r"P_\mathrm{hitch}", P_hitch)


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

**Lower critical population size**: $P_\mathrm{hitch}$ equals the neutral expectation $x_0$ exactly at $N = N_\mathrm{crit}^{-} = \delta / u_m$, whatever $x_0$.
For a single invader ($x_0 = 1/N$) in a large population, $P_\mathrm{hitch}$ reduces to $u_m / \delta$, eq. 1c of Raynes et al. (2018).


In [12]:
assert is_zero(P_hitch.subs(N, N_lower) - x_0)
invader = sp.limit(P_hitch.subs(x_0, 1 / N), N, sp.oo)
show(r"\lim_{N \to \infty} P_\mathrm{hitch}\big|_{x_0 = 1/N}", invader)
invader_lead = sp.series(invader, u_m, 0, 2).removeO()
show(
    r"\lim_{N \to \infty} P_\mathrm{hitch}\big|_{x_0 = 1/N}",
    invader_lead,
    approx=True,
)
assert is_zero(invader_lead - u_m / delta)


<IPython.core.display.Math object>

<IPython.core.display.Math object>

## 3. Sweeps

Lineages of a class are founded at per-capita rate $u$, Poisson in space and time; $V(\tau)$ is the number of individuals a lineage of age $\tau$ has reached.
A location is unswept at time $t$ if no earlier-founded lineage has reached it, and the mean sweep time integrates that probability:

$$S(t; u) = \exp\left[-u \int_0^t V(\tau)\, d\tau\right], \qquad T(u) = \int_0^\infty S(t; u)\, dt.$$

### Well-mixed

An established lineage behaves as if it began with $1/p$ copies and grows exponentially at rate $r$: $V(\tau) = e^{r \tau} / p$.


In [13]:
reach_mixed = sp.simplify(sp.integrate(sp.exp(r * tau) / p, (tau, 0, t)))
show(r"\int_0^t V(\tau)\, d\tau", reach_mixed)
S_mixed = sp.exp(-u * reach_mixed)
show("S(t; u)", S_mixed)


<IPython.core.display.Math object>

<IPython.core.display.Math object>

Writing $a = u / (p\, r)$, the sweep time is an exponential integral, $T = e^{a} E_1(a) / r$ with $E_1(a) = -\mathrm{Ei}(-a)$.
Since $a \ll 1$ (supply is far slower than growth), expanding to leading order gives the closed form.


In [14]:
a = sp.symbols("a", positive=True)
T_mixed_exact = sp.integrate(
    sp.exp(-a * (sp.exp(r * t) - 1)), (t, 0, sp.oo), conds="none"
)
show("T(u)", T_mixed_exact)

T_mixed_lead = sp.series(r * T_mixed_exact, a, 0, 1).removeO() / r
T_mixed = (sp.log(p * r / u) - sp.EulerGamma) / r
show(r"T(u)", T_mixed, approx=True)
assert is_zero(T_mixed_lead.subs(a, u / (p * r)) - T_mixed)


<IPython.core.display.Math object>

<IPython.core.display.Math object>

### Two dimensions

After a local delay $\tau_0$ a lineage spreads as a disc at wave speed $v$ through density $\rho$: $V(\tau) = \rho\, \pi\, v^2 (\tau - \tau_0)^2$ for $\tau > \tau_0$.
Nothing is swept before $\tau_0$, so $T = \tau_0 + \int_0^\infty e^{-c\, x^3} dx$ with $x = t - \tau_0$.


In [15]:
reach_2d = sp.factor(
    sp.integrate(rho * sp.pi * v**2 * (tau - tau_0) ** 2, (tau, tau_0, t))
)
show(r"\int_{\tau_0}^t V(\tau)\, d\tau", reach_2d)

c = sp.symbols("c", positive=True)
c_2d = sp.pi * u * rho * v**2 / 3
assert is_zero(u * reach_2d - (c * x**3).subs({c: c_2d, x: t - tau_0}))

tail = sp.integrate(sp.exp(-c * x**3), (x, 0, sp.oo))
show(r"\int_0^\infty e^{-c x^3}\, dx", tail)
assert is_zero(tail - sp.gamma(sp.Rational(4, 3)) / c ** sp.Rational(1, 3))

T_2d = tau_0 + sp.gamma(sp.Rational(4, 3)) * (
    3 / (sp.pi * u * rho * v**2)
) ** sp.Rational(1, 3)
assert is_zero(tau_0 + tail.subs(c, c_2d) - T_2d)
show("T(u)", T_2d)


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### Lattice of demes

A lineage grows from an effective $1/p$ copies to fill its founding deme of $n$ individuals in the fill time $t_\mathrm{fill}$, then advances one deme per $t_\mathrm{fill}$.
Measuring distance in demes: $\rho = n$, $\tau_0 = t_\mathrm{fill}$, $v = 1 / t_\mathrm{fill}$.


In [16]:
(fill_expr,) = sp.solve(sp.Eq(sp.exp(r * t) / p, n), t)
show(r"t_\mathrm{fill}", fill_expr)
assert is_zero(fill_expr - sp.log(n * p) / r)

lattice = {rho: n, tau_0: t_f, v: 1 / t_f}
T_lattice = t_f + sp.gamma(sp.Rational(4, 3)) * (
    3 * t_f**2 / (sp.pi * u * n)
) ** sp.Rational(1, 3)
assert is_zero(T_2d.subs(lattice) - T_lattice)
show("T(u)", T_lattice)


<IPython.core.display.Math object>

<IPython.core.display.Math object>

### Scaling in $d$ dimensions

With $V \propto (v \tau)^d$ the exponent is $c\, t^{d+1}$ with $c \propto u\, v^d$, so $T \propto (u\, v^d)^{-1/(d+1)}$ (cf. Ralph & Coop 2010).


In [17]:
for dim in (1, 2, 3):
    tail_d = sp.integrate(sp.exp(-c * x ** (dim + 1)), (x, 0, sp.oo))
    expected = sp.gamma(1 + sp.Rational(1, dim + 1)) * c ** sp.Rational(
        -1, dim + 1
    )
    assert is_zero(tail_d - expected)
    show(
        rf"d = {dim}: \quad \int_0^\infty e^{{-c x^{dim + 1}}}\, dx", expected
    )


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## 4. Catch-up

### Share law

Mutators and nonmutators of equal fitness compete on equal terms, so class $k + 1$ is divided in proportion to the rates at which each type founds it from class $k$.
If nonmutators hold share $a_k$ (odds $o_k = a_k / (1 - a_k)$) of class $k$, they found class $k + 1$ at rate $u_n a_k$ against mutators' $u_m (1 - a_k)$: the odds fall by $u_n / u_m$ per class.


In [18]:
k = sp.symbols("k", integer=True, nonnegative=True)
a_k, o_0 = sp.symbols("a_k o_0", positive=True)
odds_next = (u_n * a_k) / (u_m * (1 - a_k))
assert is_zero(odds_next - (u_n / u_m) * a_k / (1 - a_k))

o = sp.Function("o")
odds = sp.rsolve(o(k + 1) - (u_n / u_m) * o(k), o(k), {o(0): o_0})
show("o_k", odds)
assert is_zero(odds - o_0 * (u_n / u_m) ** k)


<IPython.core.display.Math object>

The starting odds are $(1 - x_0) / x_0$, multiplied by $P_\mathrm{hitch}$ to condition on mutators having escaped (one for a start at appreciable frequency in a large population):

$$o_k = \frac{1 - x_0}{x_0}\, P_\mathrm{hitch} \left(\frac{u_n}{u_m}\right)^k, \qquad a_k = \frac{o_k}{1 + o_k}.$$

### Catch-up probability and upper critical population size

Nonmutators one step below the peak, $N a_{K-1}$ of them, found lineages at the peak at rate $u_n$ each for the duration $T$ of mutators' final sweep.
$P_\mathrm{catchup} = 1 - \exp(-N u_n a_{K-1} T)$ is one minus the Poisson zero term.
Mutators are favored while $1 - P_\mathrm{catchup} > x_0$; solving for equality gives $N_\mathrm{crit}^{+}$.


In [19]:
a_peak = sp.symbols("a_{K-1}", positive=True)
P_catchup = 1 - sp.exp(-N * u_n * a_peak * T)
(N_upper,) = sp.solve(sp.Eq(1 - P_catchup, x_0), N)
show(r"N_\mathrm{crit}^{+}", N_upper)
assert is_zero(N_upper - sp.log(1 / x_0) / (u_n * a_peak * T))


<IPython.core.display.Math object>

For small odds $a_{K-1} \approx o_{K-1}$, so each further available mutation raises $N_\mathrm{crit}^{+}$ by the factor $u_m / u_n$.


In [20]:
N_upper_K = N_upper.subs(a_peak, o_0 * (u_n / u_m) ** (K - 1))
ratio = sp.simplify(N_upper_K.subs(K, K + 1) / N_upper_K)
show(r"N_\mathrm{crit}^{+}(K + 1) / N_\mathrm{crit}^{+}(K)", ratio)
assert is_zero(ratio - u_m / u_n)


<IPython.core.display.Math object>

## 5. Mutators arising *de novo*

### Standing mutator frequency

Mutators are supplied at rate $\mu$ and lost to their load at rate $\delta$ and to nonmutator sweeps at rate $1 / T_n$: $\dot{x} = \mu - (\delta + 1 / T_n)\, x$.


In [21]:
(xbar_expr,) = sp.solve(mu - (delta + 1 / T_n) * x, x)
show(r"\bar{x}", sp.simplify(xbar_expr))
assert is_zero(xbar_expr - mu / (delta + 1 / T_n))

# limits: sweeps rare, and sweeps frequent
assert sp.limit(xbar_expr, T_n, sp.oo) == mu / delta
assert sp.limit(xbar_expr / (mu * T_n), delta, 0) == 1


<IPython.core.display.Math object>

### Small populations

By the share law at start frequency $\bar{x} \ll 1$, a given sweep is founded by mutators at rate $\bar{x}\, u_m$ against nonmutators' $u_n$.
Mutators fix if any of the $K$ sweeps is theirs.


In [22]:
P_sweep = xbar * u_m / (u_n + xbar * u_m)
show(r"P_\mathrm{sweep}", P_sweep)
P_fix_small = 1 - (1 - P_sweep) ** K
show(r"P_\mathrm{fix}", P_fix_small)
# rare mutators: about K chances at P_sweep each
assert is_zero(
    sp.series(P_fix_small.subs(P_sweep, y), y, 0, 2).removeO() - K * y
)


<IPython.core.display.Math object>

<IPython.core.display.Math object>

### Critical adaptive potential, well-mixed

$K^*$ is where $P_\mathrm{catchup} = 1/2$, that is $N u_n a_{K^*-1} T = \ln 2$, with nonmutators' share at start frequency $\bar{x} \ll 1$ being $a_k \approx o_k = (u_n / u_m)^k / \bar{x}$.


In [23]:
share = (u_n / u_m) ** (K - 1) / xbar
(K_star_mixed,) = sp.solve(sp.Eq(N * u_n * share * T, sp.log(2)), K)
target = 1 + sp.log(N * u_n * T / (xbar * sp.log(2))) / sp.log(u_m / u_n)
assert is_zero(K_star_mixed - target)
show("K^*", target)


<IPython.core.display.Math object>

### Critical adaptive potential and peak prevalence, 2D

Mutator domains are founded at per-capita rate $\bar{x}\, u_m$ and nonmutators reach the peak after $K$ sweeps of $T_n$ each.
$K^*$ is where $\ln 2$ of the $N / n$ demes are expected to remain unreached: $(N / n)\, S(K^* T_n;\, \bar{x} u_m) = \ln 2$.


In [24]:
S_2d = sp.exp(-u * reach_2d)
show("S(t; u)", S_2d)

# solve for the cube of elapsed spread time, w^3 = (K T_n - tau_0)^3
cube = sp.symbols("w^3", positive=True)
unreached = (N / n) * S_2d.subs({t: sp.cbrt(cube) + tau_0, u: xbar * u_m})
(cube_star,) = sp.solve(sp.Eq(unreached, sp.log(2)), cube)
K_star_2d = (tau_0 + sp.cbrt(cube_star)) / T_n
target = (
    tau_0
    + (3 * sp.log(N / (n * sp.log(2))) / (sp.pi * xbar * u_m * rho * v**2))
    ** sp.Rational(1, 3)
) / T_n
assert is_zero(cube_star - (target * T_n - tau_0) ** 3)
show("K^*", target)

prevalence = 1 - S_2d.subs({t: K * T_n, u: xbar * u_m})
show(r"\text{peak prevalence}", prevalence)


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## 6. Numerical check against `pylib.theory`

Substituting simulation parameters into the symbolic results reproduces the values computed by the library.


In [25]:
params = calc_load_params()
sim = {s: 0.1, lam_b: 1e-6, lam_d: 1e-4, M: 100.0, U: 100.0 * 1e-4}

delta_val = float(delta_expr.subs(sim))
q0_val = float(q0_expr.subs(sim))
p_val = float(p_expr.subs(sim))
u_n_val = sim[lam_b] * p_val
u_m_val = sim[M] * sim[lam_b] * q0_val * p_val
r_val = sim[s] - delta_val
vals = {delta: delta_val, p: p_val, u_n: u_n_val, u_m: u_m_val, r: r_val}

assert np.isclose(delta_val, params["delta"])
assert np.isclose(q0_val, params["q0"])
assert np.isclose(p_val, params["pi_mut"])
assert np.isclose(u_n_val, params["supply_non"])
assert np.isclose(u_m_val, params["supply_mut"])
assert np.isclose(float(N_lower.subs(vals)), params["N_hitch"])
{str(key): val for key, val in vals.items()}


{'delta': 0.00985017125066523,
 'p': 0.18126924692201815,
 'u_n': 1.8126924692201815e-07,
 'u_m': 1.6323265549164637e-05,
 'r': 0.09014982874933478}

In [26]:
# escape
for N_val, x0_val in (100, 0.5), (300, 0.5), (1000, 0.5), (10**4, 0.01):
    at = {**vals, N: N_val, x_0: x0_val}
    assert np.isclose(
        float(P_drift.subs(at)), calc_drift_fixprob(N_val, x0_val)
    )
    assert np.isclose(
        float(P_hitch.subs(at)),
        calc_hitchhike_prob_closed(N_val, x0_val),
    )


In [27]:
# sweep times: closed forms against the library, and against direct
# numerical integration of S(t)
T_mut = float(T_mixed.subs({**vals, u: u_m_val}))
assert np.isclose(T_mut, calc_sweep_time(u_m_val, r_val, p_val))
exact = float(
    T_mixed_exact.subs({a: u_m_val / (p_val * r_val), r: r_val}).evalf()
)
print(f"well-mixed T(u_m): leading order {T_mut:.2f}, exact {exact:.2f}")
assert np.isclose(T_mut, exact, rtol=1e-2)

for n_val in 32, 256, 2048:
    fill_val = float(fill_expr.subs({**vals, n: n_val}))
    assert np.isclose(fill_val, calc_fill_time(n_val, r_val, p_val))
    at = {**vals, u: u_m_val, n: n_val, t_f: fill_val}
    T_val = float(T_lattice.subs(at))
    assert np.isclose(T_val, calc_sweep_time(u_m_val, r_val, p_val, n_val))
    S_num = sp.lambdify(
        t, S_2d.subs({**at, rho: n_val, tau_0: fill_val, v: 1 / fill_val})
    )
    numeric = fill_val + float(
        sp.Integral(
            S_2d.subs({**at, rho: n_val, tau_0: fill_val, v: 1 / fill_val}),
            (t, fill_val, sp.oo),
        ).evalf()
    )
    assert np.isclose(T_val, numeric)
    assert np.isclose(
        S_num(2 * fill_val),
        calc_unswept_prob(2 * fill_val, u_m_val, r_val, p_val, n_val),
    )
    print(f"{n_val}-agent demes T(u_m): {T_val:.1f}")


well-mixed T(u_m): leading order 70.23, exact 70.32
32-agent demes T(u_m): 98.6
256-agent demes T(u_m): 109.1
2048-agent demes T(u_m): 110.0


In [28]:
# upper critical population size, 50/50 start
for n_val in None, 256:
    T_val = calc_sweep_time(u_m_val, r_val, p_val, n_val)
    for K_val in 1, 2, 3, 4:
        odds_val = (u_n_val / u_m_val) ** (K_val - 1)  # x_0 = 1/2, escaped
        at = {
            u_n: u_n_val,
            a_peak: odds_val / (1 + odds_val),
            T: T_val,
            x_0: 0.5,
        }
        assert np.isclose(
            float(N_upper.subs(at)),
            calc_ncrit_closed(K_val, "upper", deme_size=n_val),
            rtol=1e-6,
        )


### Whole expressions

The pieces above composed into the full fixation probability, $P_\mathrm{fix} = P_\mathrm{escape}\,(1 - P_\mathrm{catchup})$, as one symbolic expression in the raw parameters $N$, $K$, $x_0$, $s$, $\lambda_b$, $\lambda_d$, $M$ (and deme size $n$), compared with `calc_mutator_fixprob_closed` across population sizes, adaptive potentials, start frequencies and structures.


In [29]:
raw = {
    delta: delta_expr,
    p: p_expr,
    r: s - delta_expr,
    u_n: lam_b * p_expr,
    u_m: M * lam_b * q0_expr.subs(U, M * lam_d) * p_expr,
}
P_escape = P_drift + (1 - P_drift) * P_hitch
odds_K = (1 - x_0) / x_0 * P_hitch * (u_n / u_m) ** (K - 1)
P_catchup_K = P_catchup.subs(a_peak, odds_K / (1 + odds_K))
P_fix = P_escape * (1 - P_catchup_K)

T_mixed_raw = T_mixed.subs(u, u_m).subs(raw)
T_lattice_raw = T_lattice.subs(t_f, fill_expr).subs(u, u_m).subs(raw)
P_fix_mixed = P_fix.subs(T, T_mixed_raw).subs(raw)
P_fix_lattice = P_fix.subs(T, T_lattice_raw).subs(raw)
show(r"P_\mathrm{fix}", P_fix)


<IPython.core.display.Math object>

In [30]:
sim_raw = {s: 0.1, lam_b: 1e-6, lam_d: 1e-4, M: 100.0}
args = (N, K, x_0, n)
f_fix = {
    None: sp.lambdify(args, P_fix_mixed.subs(sim_raw), "mpmath"),
    256: sp.lambdify(args, P_fix_lattice.subs(sim_raw), "mpmath"),
}
f_esc = sp.lambdify(args, P_escape.subs(raw).subs(sim_raw), "mpmath")
f_catch = {
    None: sp.lambdify(
        args,
        P_catchup_K.subs(T, T_mixed_raw).subs(raw).subs(sim_raw),
        "mpmath",
    ),
    256: sp.lambdify(
        args,
        P_catchup_K.subs(T, T_lattice_raw).subs(raw).subs(sim_raw),
        "mpmath",
    ),
}

num_checked = 0
for n_val in None, 256:
    for x0_val in 0.5, 0.1, 0.01:
        for K_val in 1, 2, 3, 4, 5:
            # the library treats N <= n as a single well-mixed deme
            for N_val in np.logspace(1 if n_val is None else 3, 11, 41):
                at = (N_val, K_val, x0_val, n_val or 1)
                assert np.isclose(
                    float(f_esc(*at)),
                    calc_escape_prob_closed(N_val, x0_val),
                    rtol=1e-6,
                    atol=1e-12,
                )
                assert np.isclose(
                    float(f_catch[n_val](*at)),
                    calc_catchup_prob_closed(
                        N_val, K_val, x0_val, deme_size=n_val
                    )[0],
                    rtol=1e-6,
                    atol=1e-12,
                )
                assert np.isclose(
                    float(f_fix[n_val](*at)),
                    calc_mutator_fixprob_closed(
                        N_val, K_val, x0_val, deme_size=n_val
                    ),
                    rtol=1e-6,
                    atol=1e-12,
                )
                num_checked += 1
print(f"P_escape, P_catchup and P_fix match at {num_checked} parameter points")


P_escape, P_catchup and P_fix match at 1230 parameter points


In [31]:
# de novo
mu_val = 1e-6
for n_val in None, 256:
    Tn_val = calc_sweep_time(u_n_val, sim[s], p_val, n_val)
    xbar_val = float(
        xbar_expr.subs({mu: mu_val, delta: delta_val, T_n: Tn_val})
    )
    assert np.isclose(xbar_val, calc_denovo_mutator_frequency(Tn_val, mu_val))

assert np.isclose(
    float(
        P_sweep.subs({xbar: mu_val / delta_val, u_n: u_n_val, u_m: u_m_val})
    ),
    calc_denovo_step_prob(),
)

N_val = 2**24
Tn_val = calc_sweep_time(u_n_val, sim[s], p_val)
at = {
    N: N_val,
    u_n: u_n_val,
    u_m: u_m_val,
    T: calc_sweep_time(u_m_val, r_val, p_val),
    xbar: calc_denovo_mutator_frequency(Tn_val, mu_val),
}
assert np.isclose(
    float(K_star_mixed.subs(at)),
    calc_denovo_critical_potential_closed(N_val),
)

n_val = 256
fill_val = calc_fill_time(n_val, r_val, p_val)
Tn_val = calc_sweep_time(u_n_val, sim[s], p_val, n_val)
at = {
    N: N_val,
    n: n_val,
    rho: n_val,
    tau_0: fill_val,
    v: 1 / fill_val,
    u_m: u_m_val,
    T_n: Tn_val,
    xbar: calc_denovo_mutator_frequency(Tn_val, mu_val),
}
assert np.isclose(
    float(K_star_2d.subs(at)),
    calc_denovo_critical_potential_closed(N_val, n_val),
)
for K_val in 1, 5, 10, 20:
    assert np.isclose(
        float(prevalence.subs({**at, K: K_val})),
        calc_denovo_peak_prevalence_closed(K_val, n_val),
    )
print("all symbolic results match pylib.theory")


all symbolic results match pylib.theory
